# 第16章：输出解析器与 LCEL

本章介绍 LangChain 中的输出解析器（Output Parsers）和 LangChain 表达式语言（LCEL）。输出解析器帮助将 LLM 的文本输出转换为结构化数据，而 LCEL 提供了一种简洁的方式来组合 LangChain 组件。

**核心知识点**：
- 输出解析器的类型和使用
- Pydantic 结构化输出
- LCEL 表达式语言基础
- 链式调用和组合模式

## 学习目标与环境准备

**学习目标**：
1. 理解输出解析器的作用和类型
2. 掌握使用 Pydantic 定义结构化输出
3. 学习 LCEL 语法和链式调用
4. 能够组合多个组件构建复杂工作流

**环境准备**：本章通过模拟代码实现，无需安装额外依赖。

## 16.1 基础输出解析器

让我们从最基础的输出解析器开始，包括字符串解析器、JSON解析器等。

In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional, Type
import json
import re


class BaseOutputParser(ABC):
    @abstractmethod
    def parse(self, text: str) -> Any:
        pass
    
    def get_format_instructions(self) -> str:
        return ""


class StrOutputParser(BaseOutputParser):
    def parse(self, text: str) -> str:
        return text.strip()
    
    def get_format_instructions(self) -> str:
        return "返回纯文本内容。"


class JsonOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Dict:
        try:
            json_match = re.search(r'\{[\s\S]*\}', text)
            if json_match:
                return json.loads(json_match.group())
            return json.loads(text)
        except Exception as e:
            raise ValueError(f"解析JSON失败: {e}")
    
    def get_format_instructions(self) -> str:
        return "请以JSON格式返回结果。"


class CommaSeparatedListOutputParser(BaseOutputParser):
    def parse(self, text: str) -> List[str]:
        items = re.split(r'[,，\n]', text)
        return [item.strip() for item in items if item.strip()]
    
    def get_format_instructions(self) -> str:
        return "请用逗号分隔返回多个项目。"


print("=== 基础输出解析器 ===")

# 测试字符串解析器
str_parser = StrOutputParser()
str_result = str_parser.parse("  Hello, World!  \n")
print(f"StrOutputParser: '{str_result}'")

# 测试JSON解析器
json_parser = JsonOutputParser()
json_input = '''这里是一些文本
{"name": "张三", "age": 25, "city": "北京"}
其他内容'''
json_result = json_parser.parse(json_input)
print(f"JsonOutputParser: {json_result}")

# 测试逗号分隔列表解析器
list_parser = CommaSeparatedListOutputParser()
list_result = list_parser.parse("苹果, 香蕉, 橙子, 葡萄")
print(f"CommaSeparatedListOutputParser: {list_result}")

## 16.2 模拟 Pydantic 结构化输出

Pydantic 是一个强大的数据验证库，LangChain 使用它来定义结构化输出模式。

In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class Person:
    name: str
    age: int
    city: str


@dataclass
class Movie:
    title: str
    year: int
    genre: List[str] = field(default_factory=list)
    rating: float = 0.0


class PydanticOutputParser(BaseOutputParser):
    def __init__(self, pydantic_class: Type):
        self.pydantic_class = pydantic_class
    
    def parse(self, text: str) -> Any:
        json_parser = JsonOutputParser()
        data = json_parser.parse(text)
        return self.pydantic_class(**data)
    
    def get_format_instructions(self) -> str:
        fields = self.pydantic_class.__dataclass_fields__
        field_desc = []
        for name, field_info in fields.items():
            field_desc.append(f'  "{name}": <{field_info.type.__name__}>')
        
        return f'''
请以以下JSON格式返回结果：
{{
{',\n'.join(field_desc)}
}}'''


print("=== Pydantic 结构化输出 ===")

# 测试 Person 解析
person_parser = PydanticOutputParser(Person)
print("格式说明:", person_parser.get_format_instructions())

person_input = '''
{"name": "李四", "age": 30, "city": "上海"}
'''
person = person_parser.parse(person_input)
print(f"\n解析结果: {person}")
print(f"  姓名: {person.name}")
print(f"  年龄: {person.age}")
print(f"  城市: {person.city}")

# 测试 Movie 解析
movie_parser = PydanticOutputParser(Movie)
movie_input = '''
{"title": "肖申克的救赎", "year": 1994, "genre": ["剧情", "犯罪"], "rating": 9.7}
'''
movie = movie_parser.parse(movie_input)
print(f"\n电影信息: {movie}")
print(f"  标题: {movie.title}")
print(f"  年份: {movie.year}")
print(f"  类型: {movie.genre}")
print(f"  评分: {movie.rating}")

## 16.3 更多输出解析器

让我们实现一些更实用的输出解析器，比如日期解析器、枚举解析器等。

In [ ]:
from datetime import datetime
from enum import Enum


class Sentiment(Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"


class EnumOutputParser(BaseOutputParser):
    def __init__(self, enum_class: Type[Enum]):
        self.enum_class = enum_class
    
    def parse(self, text: str) -> Enum:
        text = text.strip().lower()
        for member in self.enum_class:
            if member.value.lower() in text or member.name.lower() in text:
                return member
        raise ValueError(f"无法解析为 {self.enum_class.__name__}: {text}")
    
    def get_format_instructions(self) -> str:
        options = [m.value for m in self.enum_class]
        return f"请从以下选项中选择一个返回: {', '.join(options)}"


class DatetimeOutputParser(BaseOutputParser):
    def __init__(self, format: str = "%Y-%m-%d"):
        self.format = format
    
    def parse(self, text: str) -> datetime:
        date_match = re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', text)
        if date_match:
            date_str = date_match.group().replace('/', '-')
            return datetime.strptime(date_str, "%Y-%m-%d")
        raise ValueError(f"无法解析日期: {text}")
    
    def get_format_instructions(self) -> str:
        return f"请以 YYYY-MM-DD 格式返回日期。"


print("=== 更多输出解析器 ===")

# 测试情感解析器
sentiment_parser = EnumOutputParser(Sentiment)
print("格式说明:", sentiment_parser.get_format_instructions())

sentiments = [
    "这个产品太棒了，非常满意！",
    "服务很差，不会再来了",
    "一般般，没什么特别的"
]

for text in sentiments:
    result = sentiment_parser.parse(text)
    print(f"\n文本: '{text}'")
    print(f"情感: {result} ({result.value})")

# 测试日期解析器
date_parser = DatetimeOutputParser()
date_input = "会议将在 2024-06-15 举行，大家做好准备。"
date_result = date_parser.parse(date_input)
print(f"\n日期解析:")
print(f"输入: {date_input}")
print(f"解析结果: {date_result}")
print(f"格式化: {date_result.strftime('%Y年%m月%d日')}")

## 16.4 模拟 LLM 和提示模板

让我们创建一个简单的模拟 LLM 和提示模板，以便演示完整的工作流。

In [ ]:
class PromptTemplate:
    def __init__(self, template: str, input_variables: List[str] = None):
        self.template = template
        self.input_variables = input_variables or re.findall(r'\{(\w+)\}', template)
    
    def format(self, **kwargs) -> str:
        return self.template.format(**kwargs)


class MockLLM:
    def __init__(self, model_name: str = "mock-model"):
        self.model_name = model_name
    
    def invoke(self, prompt: str) -> str:
        print(f"\n[LLM] 收到提示词...")
        
        if "人物信息" in prompt or "Person" in prompt:
            return '''
{"name": "王五", "age": 28, "city": "广州"}
'''
        elif "电影" in prompt or "movie" in prompt.lower():
            return '''
{"title": "盗梦空间", "year": 2010, "genre": ["科幻", "动作", "悬疑"], "rating": 9.3}
'''
        elif "情感" in prompt or "sentiment" in prompt.lower():
            if "好" in prompt or "棒" in prompt:
                return "positive"
            elif "差" in prompt or "坏" in prompt:
                return "negative"
            return "neutral"
        elif "列表" in prompt or "list" in prompt.lower():
            return "Python, JavaScript, Java, Go, Rust"
        else:
            return "这是一个模拟的LLM响应。"

    def __call__(self, prompt: str) -> str:
        return self.invoke(prompt)


print("=== 模拟组件 ===")

# 测试提示模板
prompt = PromptTemplate(
    template="请为 {topic} 创建一个简介，不超过 {max_words} 个字。",
    input_variables=["topic", "max_words"]
)
formatted_prompt = prompt.format(topic="人工智能", max_words=100)
print(f"格式化提示词: {formatted_prompt}")

# 测试 LLM
llm = MockLLM()
response = llm.invoke("生成人物信息")
print(f"\nLLM响应: {response}")

## 16.5 LCEL 基础 - 管道操作符

LangChain 表达式语言（LCEL）使用 `|` 操作符来组合组件。让我们模拟这个功能。

In [ ]:
class Runnable:
    def __init__(self, func):
        self.func = func
    
    def invoke(self, input_data: Any) -> Any:
        return self.func(input_data)
    
    def __or__(self, other: 'Runnable') -> 'RunnableSequence':
        if isinstance(other, RunnableSequence):
            return RunnableSequence([self] + other.steps)
        return RunnableSequence([self, other])


class RunnableSequence:
    def __init__(self, steps: List[Runnable]):
        self.steps = steps
    
    def invoke(self, input_data: Any) -> Any:
        result = input_data
        for i, step in enumerate(self.steps):
            print(f"[步骤 {i+1}] 执行: {step.func.__name__}")
            result = step.invoke(result)
        return result
    
    def __or__(self, other: Runnable) -> 'RunnableSequence':
        return RunnableSequence(self.steps + [other])


print("=== LCEL 管道操作 ===")

# 创建一些简单的可运行组件
def add_one(x):
    return x + 1

def multiply_by_two(x):
    return x * 2

def square(x):
    return x * x

# 包装为 Runnable
r_add_one = Runnable(add_one)
r_multiply = Runnable(multiply_by_two)
r_square = Runnable(square)

# 组合管道
chain = r_add_one | r_multiply | r_square

# 执行
result = chain.invoke(3)
print(f"\n输入: 3")
print(f"结果: {result}")
print(f"计算过程: ((3 + 1) * 2)² = {result}")

# 另一个例子：字符串处理
def to_uppercase(s):
    return s.upper()

def add_exclamation(s):
    return s + "!"

def reverse_string(s):
    return s[::-1]

string_chain = Runnable(to_uppercase) | Runnable(add_exclamation) | Runnable(reverse_string)
result2 = string_chain.invoke("hello")
print(f"\n字符串链: 'hello' -> '{result2}'")

## 16.6 完整工作流：提示词 + LLM + 输出解析器

现在让我们把所有组件组合起来，演示一个完整的 LCEL 工作流。

In [ ]:
print("=== 完整 LCEL 工作流 ===")

# 1. 创建提示模板
person_prompt = PromptTemplate(
    template="请生成一个虚拟人物信息，包含姓名、年龄和城市。{format_instructions}",
    input_variables=["format_instructions"]
)

# 2. 创建输出解析器
person_parser = PydanticOutputParser(Person)

# 3. 创建 LLM
llm = MockLLM()

# 4. 组合成链
def create_prompt(inputs):
    format_instructions = person_parser.get_format_instructions()
    return person_prompt.format(format_instructions=format_instructions)

def parse_output(text):
    return person_parser.parse(text)

# 使用 LCEL 风格组合
person_chain = (
    Runnable(create_prompt)
    | Runnable(llm.invoke)
    | Runnable(parse_output)
)

# 执行链
print("\n--- 开始执行人物信息生成链 ---")
person_result = person_chain.invoke({})
print(f"\n最终结果:")
print(f"  类型: {type(person_result)}")
print(f"  数据: {person_result}")

# 另一个例子：电影信息链
print("\n" + "="*60)
print("--- 电影信息生成链 ---")

movie_prompt = PromptTemplate(
    template="请推荐一部好看的电影。{format_instructions}",
    input_variables=["format_instructions"]
)
movie_parser = PydanticOutputParser(Movie)

movie_chain = (
    Runnable(lambda _: movie_prompt.format(format_instructions=movie_parser.get_format_instructions()))
    | Runnable(llm.invoke)
    | Runnable(movie_parser.parse)
)

movie_result = movie_chain.invoke({})
print(f"\n电影推荐:")
print(f"  标题: {movie_result.title}")
print(f"  年份: {movie_result.year}")
print(f"  类型: {', '.join(movie_result.genre)}")
print(f"  评分: {movie_result.rating}")

## 16.7 RunnablePassthrough 和 RunnableMap

让我们模拟 LCEL 中的另外两个重要概念：RunnablePassthrough 和 RunnableMap。

In [ ]:
class RunnablePassthrough:
    def invoke(self, input_data: Any) -> Any:
        return input_data
    
    def __or__(self, other):
        if isinstance(other, RunnableSequence):
            return RunnableSequence([Runnable(lambda x: x)] + other.steps)
        return RunnableSequence([Runnable(lambda x: x), other])


class RunnableMap:
    def __init__(self, **runnables):
        self.runnables = runnables
    
    def invoke(self, input_data: Any) -> Dict:
        result = {}
        for key, runnable in self.runnables.items():
            if hasattr(runnable, 'invoke'):
                result[key] = runnable.invoke(input_data)
            else:
                result[key] = runnable
        return result


print("=== RunnablePassthrough 和 RunnableMap ===")

# 测试 RunnablePassthrough
passthrough = RunnablePassthrough()
test_data = {"name": "test", "value": 42}
print(f"RunnablePassthrough: {passthrough.invoke(test_data)}")

# 测试 RunnableMap
def get_length(s):
    return len(s)

def get_upper(s):
    return s.upper()

def get_first_char(s):
    return s[0] if s else ""

runnable_map = RunnableMap(
    original=RunnablePassthrough(),
    length=Runnable(get_length),
    uppercase=Runnable(get_upper),
    first_char=Runnable(get_first_char)
)

map_result = runnable_map.invoke("hello world")
print(f"\nRunnableMap 结果:")
for key, value in map_result.items():
    print(f"  {key}: {value}")

# 组合使用
print("\n" + "="*60)
print("--- 组合使用 ---")

complex_chain = (
    RunnablePassthrough()
    | RunnableMap(
        input=RunnablePassthrough(),
        doubled=Runnable(lambda x: x * 2),
        squared=Runnable(lambda x: x * x)
    )
)

complex_result = complex_chain.invoke(5)
print(f"输入: 5")
print(f"结果: {complex_result}")

## 16.8 实际应用：情感分析工作流

让我们创建一个更完整的实际应用示例：情感分析工作流。

In [ ]:
@dataclass
class SentimentAnalysis:
    sentiment: Sentiment
    confidence: float
    reasoning: str


print("=== 情感分析工作流 ===")

# 创建情感分析专用的 LLM
class SentimentLLM(MockLLM):
    def invoke(self, prompt: str) -> str:
        print(f"\n[情感分析LLM] 分析中...")
        if "好" in prompt or "棒" in prompt or "喜欢" in prompt:
            return '''
{"sentiment": "positive", "confidence": 0.95, "reasoning": "文本包含积极词汇，表达了满意的情绪。"}
'''
        elif "差" in prompt or "坏" in prompt or "讨厌" in prompt:
            return '''
{"sentiment": "negative", "confidence": 0.90, "reasoning": "文本包含消极词汇，表达了不满的情绪。"}
        else:
            return '''
{"sentiment": "neutral", "confidence": 0.70, "reasoning": "文本情感不明显，保持中立态度。"}
'''


# 创建情感分析解析器
class SentimentAnalysisParser(BaseOutputParser):
    def parse(self, text: str) -> SentimentAnalysis:
        json_parser = JsonOutputParser()
        data = json_parser.parse(text)
        data["sentiment"] = Sentiment(data["sentiment"])
        return SentimentAnalysis(**data)
    
    def get_format_instructions(self) -> str:
        return '''
请以JSON格式返回情感分析结果：
{
  "sentiment": "positive|negative|neutral",
  "confidence": 0.0-1.0之间的浮点数,
  "reasoning": "分析理由"
}
'''


# 创建完整的情感分析链
sentiment_prompt = PromptTemplate(
    template="分析以下文本的情感：{text}\n{format_instructions}",
    input_variables=["text", "format_instructions"]
)

sentiment_llm = SentimentLLM()
sentiment_parser = SentimentAnalysisParser()

def build_sentiment_chain(input_data):
    prompt_text = sentiment_prompt.format(
        text=input_data["text"],
        format_instructions=sentiment_parser.get_format_instructions()
    )
    llm_response = sentiment_llm.invoke(prompt_text)
    return sentiment_parser.parse(llm_response)

sentiment_chain = Runnable(build_sentiment_chain)

# 测试不同的文本
test_texts = [
    "这个产品真的很好用，我非常喜欢！",
    "服务太差了，再也不会来了。",
    "今天天气还可以。"
]

for text in test_texts:
    print("\n" + "="*60)
    print(f"分析文本: {text}")
    result = sentiment_chain.invoke({"text": text})
    print(f"\n分析结果:")
    print(f"  情感: {result.sentiment.value}")
    print(f"  置信度: {result.confidence:.1%}")
    print(f"  理由: {result.reasoning}")

## 16.9 LangChain 真实代码参考

以下是真实 LangChain 代码的示例，供你参考。

In [ ]:
print("""
=== LangChain 真实代码示例 ===

# 1. 基础导入
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

# 2. 定义 Pydantic 模型
class Person(BaseModel):
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    city: str = Field(description="城市")

# 3. 创建组件
parser = JsonOutputParser(pydantic_object=Person)
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有用的助手。{format_instructions}"),
    ("user", "生成一个虚拟人物信息")
])
llm = ChatOpenAI(model="gpt-3.5-turbo")

# 4. LCEL 组合链
chain = (
    prompt 
    | llm 
    | parser
)

# 5. 执行
result = chain.invoke({
    "format_instructions": parser.get_format_instructions()
})
print(result)

# 6. 使用 RunnableMap
from langchain_core.runnables import RunnablePassthrough, RunnableMap

map_chain = RunnableMap({
    "original": RunnablePassthrough(),
    "length": lambda x: len(x),
    "upper": lambda x: x.upper()
})

print(map_chain.invoke("hello"))

# 7. 带分支的复杂链
sentiment_prompt = ChatPromptTemplate.from_template("分析情感: {text}")
summary_prompt = ChatPromptTemplate.from_template("总结: {text}")

analysis_chain = (
    RunnableMap({
        "sentiment": sentiment_prompt | llm | StrOutputParser(),
        "summary": summary_prompt | llm | StrOutputParser(),
        "original": RunnablePassthrough()
    })
)

result = analysis_chain.invoke({"text": "这个产品很好用！"})
print(result)
""")

## 练习

1. **创建自定义输出解析器**：设计一个输出解析器，用于解析 Markdown 表格为结构化数据。

2. **构建信息提取链**：使用 LCEL 构建一个链，从文本中提取关键信息（人物、时间、地点）并返回结构化数据。

3. **多步骤工作流**：创建一个包含多个步骤的复杂链，如：输入查询 → 搜索 → 摘要 → 结构化输出。

4. **真实 LangChain 实践**：安装 LangChain，使用真实的 LLM（如 Ollama）和输出解析器构建一个应用。